# 02 — Origem: MySQL / MariaDB

Mesmo fluxo do notebook 01, trocando o driver. O `lakehouse.py` ja carrega
`com.mysql:mysql-connector-j`, entao nao ha nada a instalar.

> **A stack nao inclui um servidor MySQL.** Este notebook serve para conectar num MySQL externo
> (outro container, outra maquina, um servico gerenciado). Se o banco estiver fora do Docker, use o
> IP da maquina — `localhost` aqui dentro aponta para o proprio container do Spark.

In [ ]:
from lakehouse import sessao, ler_jdbc, gravar, perfil

spark = sessao("02-mysql")

In [ ]:
ORIGEM = dict(
    tipo="mysql",
    host="192.168.0.10",      # IP ou hostname do MySQL
    porta=3306,
    banco="vendas",
    usuario="app",
    senha="TROQUE",
)

TABELA_ORIGEM = "produtos"
DESTINO = "coleta.produtos"

In [ ]:
df = ler_jdbc(spark, tabela=TABELA_ORIGEM, **ORIGEM)
perfil(df)

### Particularidades do MySQL

- **Fuso e datas.** `DATETIME` sem fuso costuma chegar deslocado. Se acontecer, force o fuso na
  conexao: `ler_jdbc(..., serverTimezone="America/Recife")`.
- **`TINYINT(1)`** vira boolean por padrao. Para manter numero:
  `ler_jdbc(..., tinyInt1isBit=False)`.
- **TLS.** Em servidor sem certificado valido, `ler_jdbc(..., useSSL=False)`. Nunca faca isso em
  banco exposto na internet.
- **MariaDB** funciona com este mesmo driver na maioria dos casos.

In [ ]:
gravar(df, DESTINO, modo="substituir")